# 📈 Predictive Modeling – Practical Session
### From Scatter Plots to Linear Regression & Association Rules

---

**Learning Objectives:**
By the end of this session, you will be able to:
- Explore relationships between two numeric variables visually
- Draw and interpret a regression line — first by hand, then with code
- Calculate and interpret a simple linear regression: slope, intercept, R²
- Judge when linear regression is appropriate and when it is not
- Understand association rules with a small market basket example

---

**Dataset:** We use the [House Prices – Advanced Regression Techniques](https://huggingface.co/datasets/michaelmallari/house-prices-advanced-regression-techniques) dataset hosted on HuggingFace.  
It contains real estate data from Ames, Iowa — house characteristics and their sale prices.

> 💡 **Recall from the lecture:** Building a model means finding a mathematical function that maps input variables (features) to an output (target). Linear regression finds the *best-fitting straight line* through the data.

---
## 📦 Step 0: Install & Import Libraries

In [ ]:
!pip install mlxtend huggingface_hub --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

print('✅ All libraries imported.')

---
## 🗂️ Part 1: Load & Prepare the Dataset

> 🎯 **Task:** Load the house prices dataset directly from HuggingFace, understand its structure, and do lightweight cleaning.

We load the file using the `hf://` path — no API token or `load_dataset` needed.

In [ ]:
# Load directly from HuggingFace as a CSV — works without authentication
HF_URL = (
    "hf://datasets/michaelmallari/"
    "house-prices-advanced-regression-techniques/train.csv"
)

df_raw = pd.read_csv(HF_URL)

print(f"Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
# Quick statistical overview of all numeric columns
df_raw.describe().round(0)

### 1.1 – Select & Rename Relevant Columns

In [ ]:
# We keep a focused subset — all columns we will use across the session
keep = {
    'GrLivArea'    : 'living_area_sqft',  # Above-ground living area (sq ft)
    'SalePrice'    : 'sale_price',         # Sale price in USD
    'YearBuilt'    : 'year_built',         # Year the house was built
    'OverallQual'  : 'overall_quality',    # Material & finish quality (1–10)
    'TotalBsmtSF'  : 'basement_sqft',      # Total basement area (sq ft)
    'GarageArea'   : 'garage_sqft',        # Garage area (sq ft)
    'LotArea'      : 'lot_area_sqft',      # Total lot size (sq ft)
    'BedroomAbvGr' : 'bedrooms',           # Bedrooms above grade
    'Fireplaces'   : 'fireplaces',         # Number of fireplaces
}

df = df_raw[list(keep.keys())].rename(columns=keep).copy()

# ── Basic cleaning ────────────────────────────────────────────────────────────
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows with missing values ({len(df)} remaining)")

# Remove extreme outliers that distort the scatter plots
df = df[df['living_area_sqft'] < 4000]
df = df[df['lot_area_sqft']    < 50000]
df = df.reset_index(drop=True)

print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

**Column reference:**

| Column | Type | Description |
|---|---|---|
| `living_area_sqft` | Numeric | Above-ground living area (sq ft) |
| `sale_price` | Numeric | Sale price of the house (USD) |
| `year_built` | Numeric | Year the house was built |
| `overall_quality` | Ordinal (1–10) | Quality of materials & finish |
| `basement_sqft` | Numeric | Basement floor area (sq ft) |
| `garage_sqft` | Numeric | Garage floor area (sq ft) |
| `lot_area_sqft` | Numeric | Total lot size (sq ft) |
| `bedrooms` | Numeric (count) | Number of bedrooms above grade |
| `fireplaces` | Numeric (count) | Number of fireplaces |

---
## 🔍 Part 2: Explore the Relationship – Living Area vs. Price

> 🎯 **Task:** Visualize the relationship before fitting any model.

> 💡 **Key idea from the lecture:** Before building a model, always *look at the data*. A scatter plot shows whether a linear relationship is even plausible.

### 2.1 – Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(df['living_area_sqft'], df['sale_price'] / 1000,
           alpha=0.35, s=25, color='#4C9BE8', edgecolors='none')

ax.set_title('Living Area vs. Sale Price', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Living Area (sq ft)', fontsize=12)
ax.set_ylabel('Sale Price ($ thousands)', fontsize=12)
ax.grid(alpha=0.25)
ax.annotate('Small & affordable', xy=(800, 90),   fontsize=9, color='#666', style='italic')
ax.annotate('Large & expensive',  xy=(2900, 350), fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.show()

corr = df['living_area_sqft'].corr(df['sale_price'])
print(f"Pearson correlation (r): {corr:.3f}")
print('\n💬 What pattern do you see? Does the relationship look linear?')

### 2.2 – Draw Your Own Regression Line (Visual Estimation)

> ✏️ **Exercise:** Before running any regression code, **estimate the line yourself**.  
> Adjust `your_slope` and `your_intercept` below until your line looks like a good fit.  
> Remember: the y-axis is in *thousands of dollars*.

**Hint:** Start with `slope = 0.10`, `intercept = 20` and tune from there.

In [ ]:
# ✏️ CHANGE THESE VALUES
your_slope     = 0.10   # ← $k per sq ft
your_intercept = 20     # ← predicted price ($k) when area = 0

x_range   = np.linspace(df['living_area_sqft'].min(), df['living_area_sqft'].max(), 200)
your_line = your_slope * x_range + your_intercept

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['living_area_sqft'], df['sale_price'] / 1000,
           alpha=0.3, s=22, color='#4C9BE8', edgecolors='none', label='Data points')
ax.plot(x_range, your_line, color='#E87A4C', linewidth=2.5,
        label=f'Your line: price = {your_slope}·area + {your_intercept}')

ax.set_title('Living Area vs. Sale Price — Your Estimated Line', fontsize=13, fontweight='bold')
ax.set_xlabel('Living Area (sq ft)')
ax.set_ylabel('Sale Price ($ thousands)')
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

y_pred_yours = your_slope * df['living_area_sqft'] + your_intercept
r2_yours = r2_score(df['sale_price'] / 1000, y_pred_yours)
print(f"Your line's R² = {r2_yours:.3f}  (1.0 = perfect, 0.0 = no better than the mean)")

---
## 📐 Part 3: Calculate Linear Regression with Code

> 🎯 **Task:** Let Python find the *optimal* slope and intercept, then interpret the output together.

> 💡 **Recall from the lecture:** The optimal line minimizes the sum of squared residuals — the total squared distance between actual data points and the predicted line.

### 3.1 – Fit the Model

In [ ]:
X = df[['living_area_sqft']].values  # 2D array required by sklearn
y = df['sale_price'].values

model = LinearRegression()
model.fit(X, y)

slope_     = model.coef_[0]
intercept_ = model.intercept_
y_pred     = model.predict(X)
r2_        = r2_score(y, y_pred)
rmse_      = np.sqrt(mean_squared_error(y, y_pred))

print('=' * 52)
print('  LINEAR REGRESSION RESULTS')
print('=' * 52)
print(f'  Equation : price = {slope_:.2f} × living_area + {intercept_:,.0f}')
print(f'  Slope    : ${slope_:.2f} per sq ft')
print(f'  Intercept: ${intercept_:,.0f}')
print(f'  R²       : {r2_:.4f}')
print(f'  RMSE     : ${rmse_:,.0f}')
print('=' * 52)

### 3.2 – Visualize: Data + Regression Line + Residuals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: scatter + line ─────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(df['living_area_sqft'], y / 1000,
           alpha=0.3, s=22, color='#4C9BE8', edgecolors='none', label='Actual prices')

x_line = np.linspace(df['living_area_sqft'].min(), df['living_area_sqft'].max(), 200)
y_line = (slope_ * x_line + intercept_) / 1000
ax.plot(x_line, y_line, color='#E84C4C', linewidth=2.5, label=f'Regression  (R²={r2_:.3f})')

# Annotate what the slope means visually
x1, x2 = 1500, 2500
y1 = (slope_ * x1 + intercept_) / 1000
y2 = (slope_ * x2 + intercept_) / 1000
ax.annotate('', xy=(x2, y2), xytext=(x1, y2),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.annotate('', xy=(x2, y2), xytext=(x2, y1),
            arrowprops=dict(arrowstyle='<->', color='#E84C4C', lw=1.5))
ax.text((x1+x2)/2, y2 - 20, '1000 sq ft', ha='center', va='top', fontsize=8, color='gray')
ax.text(x2 + 30, (y1+y2)/2, f'+${slope_*1000/1000:.0f}k', va='center', fontsize=9, color='#E84C4C')

ax.set_title('Living Area vs. Price + Regression Line', fontweight='bold')
ax.set_xlabel('Living Area (sq ft)')
ax.set_ylabel('Sale Price ($ thousands)')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

# ── Right: residuals ─────────────────────────────────────────────────────────
ax2 = axes[1]
residuals = y - y_pred
ax2.scatter(y_pred / 1000, residuals / 1000, alpha=0.3, s=22, color='#5CBF7A', edgecolors='none')
ax2.axhline(0, color='black', linewidth=1.5, linestyle='--')
ax2.set_title('Residuals Plot  (Actual − Predicted)', fontweight='bold')
ax2.set_xlabel('Predicted Price ($ thousands)')
ax2.set_ylabel('Residual ($ thousands)')
ax2.grid(alpha=0.25)
ax2.text(0.02, 0.97, 'Random scatter = good fit\nCurved pattern = not linear!',
         transform=ax2.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Linear Regression: Living Area → Sale Price', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.3 – Interpret the Output Together

In [ ]:
print('┌─────────────────────────────────────────────────────────────────────┐')
print('│  HOW TO READ THE REGRESSION OUTPUT                                  │')
print('├─────────────────────────────────────────────────────────────────────┤')
print(f'│  Equation:  price = {slope_:.2f} × living_area + {intercept_:,.0f}          │')
print('│                                                                     │')
print(f'│  SLOPE  = {slope_:.2f}                                                 │')
print(f'│  → Each additional sq ft → +${slope_:.2f} in sale price              │')
print(f'│  → For 100 sq ft more   → +${slope_*100:,.0f} expected                │')
print('│                                                                     │')
print(f'│  INTERCEPT = {intercept_:,.0f}                                      │')
print('│  → Price when area = 0  →  no meaningful real-world use             │')
print('│                                                                     │')
print(f'│  R² = {r2_:.4f}                                                     │')
print(f'│  → {r2_*100:.1f}% of price variance is explained by living area alone   │')
print(f'│  → Remaining {(1-r2_)*100:.1f}% is due to other factors                   │')
print('│                                                                     │')
print(f'│  RMSE = ${rmse_:,.0f}                                              │')
print('│  → Average prediction error in USD                                 │')
print('└─────────────────────────────────────────────────────────────────────┘')

### 3.4 – Make a Prediction

In [ ]:
# ✏️ Change this value to predict for any house size you like
my_house_sqft = 1800

predicted_price = slope_ * my_house_sqft + intercept_
print(f'A house with {my_house_sqft} sq ft of living area is predicted to sell for:')
print(f'  → ${predicted_price:,.0f}')

mask = (df['living_area_sqft'] >= my_house_sqft - 100) & \
       (df['living_area_sqft'] <= my_house_sqft + 100)
actual_nearby = df.loc[mask, 'sale_price']
print(f'\nActual prices in the dataset for houses {my_house_sqft} ±100 sq ft:')
if len(actual_nearby) > 0:
    print(f'  Min:  ${actual_nearby.min():,.0f}')
    print(f'  Max:  ${actual_nearby.max():,.0f}')
    print(f'  Mean: ${actual_nearby.mean():,.0f}  ← compare to our prediction')
else:
    print('  (No houses found in that range)')

### 3.5 – When Does Linear Regression Fail?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
np.random.seed(7)

# Case 1: Good linear fit
ax = axes[0]
ax.scatter(df['living_area_sqft'], df['sale_price']/1000, alpha=0.25, s=15, color='#4C9BE8')
m, b, r, *_ = stats.linregress(df['living_area_sqft'], df['sale_price']/1000)
xl = np.linspace(df['living_area_sqft'].min(), df['living_area_sqft'].max(), 100)
ax.plot(xl, m*xl + b, 'r-', linewidth=2)
ax.set_title(f'✅ Linear fit works\n(R²={r**2:.2f})', fontweight='bold')
ax.set_xlabel('Living Area (sq ft)'); ax.set_ylabel('Price ($k)'); ax.grid(alpha=0.2)

# Case 2: Curved/non-linear pattern
ax2 = axes[1]
x_nl = np.linspace(1, 10, 200)
y_nl = 2 * x_nl**2 + np.random.normal(0, 5, 200)
ax2.scatter(x_nl, y_nl, alpha=0.4, s=15, color='#E87A4C')
m2, b2, r2v, *_ = stats.linregress(x_nl, y_nl)
ax2.plot(x_nl, m2*x_nl + b2, 'r-', linewidth=2)
ax2.set_title(f'⚠️ Misses the curve\n(R²={r2v**2:.2f})', fontweight='bold')
ax2.set_xlabel('x'); ax2.set_ylabel('y  (quadratic truth)'); ax2.grid(alpha=0.2)

# Case 3: No relationship
ax3 = axes[2]
x_n = np.random.uniform(0, 10, 150)
y_n = np.random.uniform(0, 100, 150)
ax3.scatter(x_n, y_n, alpha=0.4, s=15, color='#5CBF7A')
m3, b3, r3, *_ = stats.linregress(x_n, y_n)
ax3.plot(x_n, m3*x_n + b3, 'r-', linewidth=2)
ax3.set_title(f'❌ No relationship\n(R²={r3**2:.2f})', fontweight='bold')
ax3.set_xlabel('x  (e.g., house ID)'); ax3.set_ylabel('y (random)'); ax3.grid(alpha=0.2)

plt.suptitle('When Does Linear Regression Work?', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💬 Takeaway:')
print('  ✅ High R² + random residuals   → linear regression is appropriate')
print('  ⚠️ Curved residual pattern      → consider polynomial regression')
print('  ❌ R² ≈ 0                       → a line is simply the wrong model')

> ✏️ **Exercise 3.6 – Interpretation**
>
> Answer in the cell below (as comments):
> 1. What does the slope mean for a house that is **200 sq ft larger**?
> 2. R² ≈ 0.50 — is this a good model? What does the remaining 50% represent?
> 3. Looking at the residuals plot: do you see any pattern? What might cause it?

In [ ]:
# ✏️ Your answers:

# 1. Price change for 200 sq ft more:

# 2. R² ≈ 0.50 — good or not? What is the remaining variance?

# 3. Residuals pattern:

---
## 👥 Part 4: Group Activity – 60 Minutes

Each group investigates a **different variable pair** from the same dataset and presents in **3 minutes**.

**Steps for every group:**
1. Create a scatter plot
2. Fit a linear regression
3. Report slope, intercept, R²
4. Discuss: *How strong is the relationship? Does it make intuitive sense?*

---

### 🔵 Group A – *Overall Quality vs. Sale Price*
**Question:** *Does higher build quality reliably predict a higher price?*  
**Types:** Ordinal (1–10) → Numeric (USD)

In [ ]:
x_A = df['overall_quality'].values
y_A = df['sale_price'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
jitter = np.random.uniform(-0.2, 0.2, len(x_A))  # spread overlapping points
ax.scatter(x_A + jitter, y_A / 1000, alpha=0.3, s=18, color='#4C9BE8', edgecolors='none')
ax.set_xlabel('Overall Quality (1–10)'); ax.set_ylabel('Sale Price ($k)')
ax.set_title('Quality vs. Price', fontweight='bold'); ax.grid(alpha=0.25)

model_A = LinearRegression().fit(x_A.reshape(-1, 1), y_A)
slope_A, intercept_A = model_A.coef_[0], model_A.intercept_
r2_A = r2_score(y_A, model_A.predict(x_A.reshape(-1, 1)))
xl_A = np.linspace(x_A.min(), x_A.max(), 100)
ax.plot(xl_A, (slope_A * xl_A + intercept_A) / 1000, 'r-', linewidth=2.5, label=f'R²={r2_A:.3f}')
ax.legend()

ax2 = axes[1]
res_A = y_A - model_A.predict(x_A.reshape(-1, 1))
ax2.scatter(model_A.predict(x_A.reshape(-1, 1)) / 1000, res_A / 1000,
            alpha=0.3, s=18, color='#5CBF7A', edgecolors='none')
ax2.axhline(0, color='black', lw=1.5, ls='--')
ax2.set_xlabel('Predicted ($k)'); ax2.set_ylabel('Residual ($k)')
ax2.set_title('Residuals', fontweight='bold'); ax2.grid(alpha=0.25)

plt.suptitle('Group A: Overall Quality → Sale Price', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Slope    : ${slope_A:,.2f} per quality point')
print(f'Intercept: ${intercept_A:,.0f}')
print(f'R²       : {r2_A:.4f}')
print(f'→ Quality 5 → 6 is associated with +${slope_A:,.0f} in price')

# ✏️ Group A interpretation:
# ...

---
### 🟠 Group B – *Year Built vs. Sale Price*
**Question:** *Are newer houses significantly more expensive?*  
**Types:** Numeric (year) → Numeric (USD)

In [ ]:
x_B = df['year_built'].values
y_B = df['sale_price'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(x_B, y_B / 1000, alpha=0.3, s=18, color='#E87A4C', edgecolors='none')
ax.set_xlabel('Year Built'); ax.set_ylabel('Sale Price ($k)')
ax.set_title('Year Built vs. Price', fontweight='bold'); ax.grid(alpha=0.25)

model_B = LinearRegression().fit(x_B.reshape(-1, 1), y_B)
slope_B, intercept_B = model_B.coef_[0], model_B.intercept_
r2_B = r2_score(y_B, model_B.predict(x_B.reshape(-1, 1)))
xl_B = np.linspace(x_B.min(), x_B.max(), 100)
ax.plot(xl_B, (slope_B * xl_B + intercept_B) / 1000, 'r-', linewidth=2.5, label=f'R²={r2_B:.3f}')
ax.legend()

ax2 = axes[1]
res_B = y_B - model_B.predict(x_B.reshape(-1, 1))
ax2.scatter(model_B.predict(x_B.reshape(-1, 1)) / 1000, res_B / 1000,
            alpha=0.3, s=18, color='#BF5C9B', edgecolors='none')
ax2.axhline(0, color='black', lw=1.5, ls='--')
ax2.set_xlabel('Predicted ($k)'); ax2.set_ylabel('Residual ($k)')
ax2.set_title('Residuals', fontweight='bold'); ax2.grid(alpha=0.25)

plt.suptitle('Group B: Year Built → Sale Price', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Slope    : ${slope_B:,.2f} per year')
print(f'Intercept: ${intercept_B:,.0f}  (extrapolation to year 0 — meaningless)')
print(f'R²       : {r2_B:.4f}')
print(f'→ A house built 10 years later → +${slope_B * 10:,.0f} in price')

# ✏️ Group B interpretation:
# ...

---
### 🟢 Group C – *Garage Area vs. Sale Price*
**Question:** *Does a bigger garage add significant value?*  
**Types:** Numeric (sq ft) → Numeric (USD)

In [ ]:
x_C = df['garage_sqft'].values
y_C = df['sale_price'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(x_C, y_C / 1000, alpha=0.3, s=18, color='#5CBF7A', edgecolors='none')
ax.set_xlabel('Garage Area (sq ft)'); ax.set_ylabel('Sale Price ($k)')
ax.set_title('Garage Area vs. Price', fontweight='bold'); ax.grid(alpha=0.25)

model_C = LinearRegression().fit(x_C.reshape(-1, 1), y_C)
slope_C, intercept_C = model_C.coef_[0], model_C.intercept_
r2_C = r2_score(y_C, model_C.predict(x_C.reshape(-1, 1)))
xl_C = np.linspace(x_C.min(), x_C.max(), 100)
ax.plot(xl_C, (slope_C * xl_C + intercept_C) / 1000, 'r-', linewidth=2.5, label=f'R²={r2_C:.3f}')
ax.legend()

ax2 = axes[1]
res_C = y_C - model_C.predict(x_C.reshape(-1, 1))
ax2.scatter(model_C.predict(x_C.reshape(-1, 1)) / 1000, res_C / 1000,
            alpha=0.3, s=18, color='#E8C94C', edgecolors='none')
ax2.axhline(0, color='black', lw=1.5, ls='--')
ax2.set_xlabel('Predicted ($k)'); ax2.set_ylabel('Residual ($k)')
ax2.set_title('Residuals', fontweight='bold'); ax2.grid(alpha=0.25)

plt.suptitle('Group C: Garage Area → Sale Price', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Slope    : ${slope_C:,.2f} per sq ft of garage')
print(f'Intercept: ${intercept_C:,.0f}')
print(f'R²       : {r2_C:.4f}')

# ✏️ Group C interpretation:
# ...

---
### 🟣 Group D – *Lot Area vs. Sale Price*
**Question:** *Does a bigger plot of land reliably mean a higher price?*  
**Bonus:** Compare your R² to the living area baseline — which is the stronger predictor?

In [ ]:
x_D = df['lot_area_sqft'].values
y_D = df['sale_price'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(x_D, y_D / 1000, alpha=0.3, s=18, color='#9B4CE8', edgecolors='none')
ax.set_xlabel('Lot Area (sq ft)'); ax.set_ylabel('Sale Price ($k)')
ax.set_title('Lot Area vs. Price', fontweight='bold'); ax.grid(alpha=0.25)

model_D = LinearRegression().fit(x_D.reshape(-1, 1), y_D)
slope_D, intercept_D = model_D.coef_[0], model_D.intercept_
r2_D = r2_score(y_D, model_D.predict(x_D.reshape(-1, 1)))
xl_D = np.linspace(x_D.min(), x_D.max(), 100)
ax.plot(xl_D, (slope_D * xl_D + intercept_D) / 1000, 'r-', linewidth=2.5, label=f'R²={r2_D:.3f}')
ax.legend()

ax2 = axes[1]
res_D = y_D - model_D.predict(x_D.reshape(-1, 1))
ax2.scatter(model_D.predict(x_D.reshape(-1, 1)) / 1000, res_D / 1000,
            alpha=0.3, s=18, color='#4CE8C9', edgecolors='none')
ax2.axhline(0, color='black', lw=1.5, ls='--')
ax2.set_xlabel('Predicted ($k)'); ax2.set_ylabel('Residual ($k)')
ax2.set_title('Residuals', fontweight='bold'); ax2.grid(alpha=0.25)

plt.suptitle('Group D: Lot Area → Sale Price', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Slope    : ${slope_D:,.4f} per sq ft of lot')
print(f'Intercept: ${intercept_D:,.0f}')
print(f'R²       : {r2_D:.4f}')
print(f'\nBonus — R² comparison:')
print(f'  Living area → R² = {r2_:.4f}')
print(f'  Lot area    → R² = {r2_D:.4f}')
print('  → Which is the better single predictor?')

# ✏️ Group D interpretation:
# ...

### 📊 Class Comparison: All Groups Side by Side

Run this cell **after** all group workspace cells have been executed.

In [ ]:
summary = pd.DataFrame({
    'Group'     : ['A', 'B', 'C', 'D', 'Baseline'],
    'X Variable': ['overall_quality', 'year_built', 'garage_sqft',
                   'lot_area_sqft', 'living_area_sqft'],
    'R²'        : [r2_A, r2_B, r2_C, r2_D, r2_]
}).sort_values('R²', ascending=False).reset_index(drop=True)
summary['R²'] = summary['R²'].round(4)

print('=== R² Ranking: Which Variable Best Predicts Sale Price? ===')
display(summary)

fig, ax = plt.subplots(figsize=(9, 5))
palette = ['#4C9BE8', '#E87A4C', '#5CBF7A', '#9B4CE8', '#E84C4C']
bars = ax.bar(summary['X Variable'], summary['R²'],
              color=palette[:len(summary)], edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, summary['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylabel('R²  (Explained Variance)', fontsize=11)
ax.set_title('Which Variable Best Explains Sale Price?', fontsize=13, fontweight='bold')
ax.set_ylim(0, min(1.0, summary['R²'].max() + 0.12))
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🛒 Part 5 (Optional): Association Rules – Market Basket Analysis

> 💡 **Recall from the lecture:** *"Customers who buy X also buy Y"* — this powers Amazon recommendations, supermarket placement, and Netflix suggestions.

| Term | Meaning | Example |
|---|---|---|
| **Support** | How often does the itemset occur? | 40% of baskets contain bread |
| **Confidence** | If X, how often also Y? | 75% of bread buyers also buy butter |
| **Lift** | Is co-occurrence beyond chance? | Lift > 1 = positive association |

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

transactions = [
    ['bread', 'butter', 'milk'],   ['bread', 'butter'],
    ['bread', 'milk', 'eggs'],     ['butter', 'milk'],
    ['bread', 'butter', 'eggs'],   ['milk', 'eggs', 'cheese'],
    ['bread', 'butter', 'milk', 'eggs'], ['butter', 'cheese'],
    ['bread', 'milk'],             ['bread', 'eggs', 'cheese'],
    ['milk', 'butter', 'cheese'],  ['bread', 'butter', 'cheese'],
    ['eggs', 'milk'],              ['bread', 'butter', 'milk', 'cheese'],
    ['bread', 'eggs'],             ['milk', 'cheese', 'butter'],
    ['bread', 'milk', 'butter'],   ['eggs', 'bread'],
    ['cheese', 'eggs', 'milk'],    ['bread', 'butter', 'milk', 'eggs', 'cheese'],
]

te = TransactionEncoder()
df_basket = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)

print(f'Transactions: {len(transactions)}')
print('\nItem frequencies:')
print(df_basket.sum().sort_values(ascending=False))

In [ ]:
frequent_itemsets = apriori(df_basket, min_support=0.30, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
print('=== Frequent Itemsets (support ≥ 30%) ===')
display(frequent_itemsets.sort_values('support', ascending=False)[['itemsets', 'support', 'length']])

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.60)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
rules[['support', 'confidence', 'lift']] = rules[['support', 'confidence', 'lift']].round(2)

print('\n=== Association Rules (confidence ≥ 60%) ===')
display(rules)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(rules['support'], rules['confidence'],
                     c=rules['lift'], cmap='RdYlGn',
                     s=rules['lift'] * 80, alpha=0.85, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Lift')

for _, row in rules.iterrows():
    label = f"{set(row['antecedents'])} → {set(row['consequents'])}"
    label = label.replace('frozenset(', '').replace(')', '')
    ax.annotate(label, xy=(row['support'], row['confidence']),
                xytext=(5, 3), textcoords='offset points', fontsize=7.5, alpha=0.85)

ax.set_xlabel('Support  (how common?)', fontsize=11)
ax.set_ylabel('Confidence  (how reliable?)', fontsize=11)
ax.set_title('Association Rules: Support vs. Confidence\n(size & color = Lift)',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print('\nTop 3 rules interpreted:')
for i, row in rules.head(3).iterrows():
    ant, con = list(row['antecedents']), list(row['consequents'])
    print(f'\n  Rule {i+1}: {ant} → {con}')
    print(f'  Support    = {row["support"]:.2f}  ({row["support"]*100:.0f}% of all transactions)')
    print(f'  Confidence = {row["confidence"]:.2f}  ({row["confidence"]*100:.0f}% of {ant} buyers also buy {con})')
    print(f'  Lift       = {row["lift"]:.2f}  ({"more" if row["lift"] > 1 else "less"} likely than chance)')

> ✏️ **Exercise 5.1**
> 1. Which rule has the highest lift? What does this mean in plain language?
> 2. Can you think of a rule with high confidence but lift ≈ 1? Give an example.
> 3. What is the fundamental difference between regression and association rules?

In [ ]:
# ✏️ Your answers:

# 1. Highest lift in plain language:

# 2. High confidence, lift ≈ 1 — example:

# 3. Regression vs. association rules:

---
## 🗣️ Class Discussion (10 Minutes)

1. **Comparing R² values:** Which variable was the best single predictor? Did this surprise you?
2. **Slope units:** How do you compare slopes that have completely different units?
3. **Correlation ≠ Causation:** Group D found lot area correlates with price. Does bigger lot *cause* higher price?
4. **Residuals:** Did any group notice a systematic pattern? What does that imply?
5. **Regression vs. Association:** When would you use each approach?

---
## ✅ Summary Checklist

- [ ] I can create a scatter plot and visually assess whether a linear relationship exists
- [ ] I can fit a simple linear regression with `LinearRegression().fit(X, y)`
- [ ] I can interpret the **slope**: "for each additional unit of X, Y changes by ..."
- [ ] I can interpret **R²**: what percentage of variance is explained?
- [ ] I can read a **residuals plot** and recognize when a linear model is not appropriate
- [ ] I understand **support**, **confidence**, and **lift** in association rules
- [ ] I can explain the difference between regression (predicting a number) and association rules (finding co-occurrence patterns)

---
**Great work! 🎉**  Next session: **multiple regression** — using more than one predictor at once.